<a href="https://colab.research.google.com/github/lamyaah/test/blob/main/CSP_astar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
!{sys.executable} -m pip install ortools moviepy imageio imageio[ffmpeg]

import time
import math
import heapq
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from moviepy.editor import VideoFileClip
from ortools.sat.python import cp_model

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 12.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 wh

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



# Environment

In [2]:
class GridEnvironment:
    def __init__(self, shape=(20, 20, 10)):
        self.shape = shape
        self.grid = np.zeros(shape, dtype=int)

    def add_obstacle_block(self, x_range, y_range, z_range):
        self.grid[
            x_range[0]:x_range[1],
            y_range[0]:y_range[1],
            z_range[0]:z_range[1]
        ] = 1

    def is_free(self, x, y, z, t, dynamic_obstacles):
        if not (
            0 <= x < self.shape[0] and
            0 <= y < self.shape[1] and
            0 <= z < self.shape[2]
        ):
            return False

        if self.grid[x, y, z] == 1:
            return False

        for obs in dynamic_obstacles:
            if obs.position_at(t) == (x, y, z):
                return False

        return True


class DynamicObstacle:
    def __init__(self, waypoints, loop=True, name="Obs"):
        self.waypoints = waypoints
        self.loop = loop
        self.name = name

    def position_at(self, t):
        if self.loop:
            return self.waypoints[t % len(self.waypoints)]
        else:
            idx = min(t, len(self.waypoints) - 1)
            return self.waypoints[idx]


class UAVPathObstacle:
    """Treats a completed UAV path as a dynamic obstacle."""
    def __init__(self, path, name):
        self.path = path
        self.name = name

    def position_at(self, t):
        idx = min(t, len(self.path) - 1)
        return self.path[idx]

# Cost, Energy, Heuristic

In [3]:
def move_cost(cur, nxt):
    dx = nxt[0] - cur[0]
    dy = nxt[1] - cur[1]
    dz = nxt[2] - cur[2]

    dist = math.sqrt(dx**2 + dy**2 + dz**2)

    # Same as your original code:
    # distance cost + climbing penalty
    return dist + max(0, dz) * 0.5


def move_energy(cur, nxt):
    dx = nxt[0] - cur[0]
    dy = nxt[1] - cur[1]
    dz = nxt[2] - cur[2]

    dist = math.sqrt(dx**2 + dy**2 + dz**2)

    if cur == nxt:
        return 0.5  # hovering energy

    climb_energy = max(0, dz) * 2.0
    descent_energy = max(0, -dz) * 0.5

    return dist + climb_energy + descent_energy


def heuristic(node, goal):
    return math.sqrt(
        sum((a - b) ** 2 for a, b in zip(node, goal))
    )


def path_cost(path):
    if not path:
        return float("inf")

    return sum(
        move_cost(path[t], path[t + 1])
        for t in range(len(path) - 1)
    )


def path_energy(path):
    if not path:
        return float("inf")

    return sum(
        move_energy(path[t], path[t + 1])
        for t in range(len(path) - 1)
    )


# Path conflict, and near miss count

In [4]:
def extend_path(path, horizon):
    path = list(path)

    while len(path) <= horizon:
        path.append(path[-1])

    return path[:horizon + 1]


def paths_conflict(path_a, path_b, horizon):
    a = extend_path(path_a, horizon)
    b = extend_path(path_b, horizon)

    for t in range(horizon + 1):
        # Same-cell collision
        if a[t] == b[t]:
            return True

    for t in range(horizon):
        # Edge-swap collision
        if a[t] == b[t + 1] and a[t + 1] == b[t]:
            return True

    return False


def near_miss_count(paths, horizon, threshold=1.5):
    extended = [extend_path(p, horizon) for p in paths]
    count = 0

    for t in range(horizon + 1):
        for i in range(len(extended)):
            for j in range(i + 1, len(extended)):
                a = extended[i][t]
                b = extended[j][t]

                d = math.sqrt(
                    (a[0] - b[0])**2 +
                    (a[1] - b[1])**2 +
                    (a[2] - b[2])**2
                )

                if d <= threshold:
                    count += 1

    return count



# Candidate generation using A*

In [5]:
def astar_spacetime_with_penalty(
    env,
    start,
    goal,
    dynamic_obstacles,
    penalty_cells=None,
    penalty_weight=3.0,
    t_max=100
):
    if penalty_cells is None:
        penalty_cells = set()

    start_state = (*start, 0)  # (x, y, z, t)

    open_heap = [(0, start_state)]
    came_from = {start_state: None}
    g_score = {start_state: 0.0}

    while open_heap:
        _, current = heapq.heappop(open_heap)

        cx, cy, cz, ct = current

        if (cx, cy, cz) == goal:
            path = []

            while current:
                path.append(current[:3])
                current = came_from[current]

            return path[::-1], g_score[(*goal, ct)], ct

        if ct >= t_max:
            continue

        moves = [
            (dx, dy, dz)
            for dx in [-1, 0, 1]
            for dy in [-1, 0, 1]
            for dz in [-1, 0, 1]
        ]

        for dx, dy, dz in moves:
            nx = cx + dx
            ny = cy + dy
            nz = cz + dz
            nt = ct + 1

            if not env.is_free(nx, ny, nz, nt, dynamic_obstacles):
                continue

            nxt = (nx, ny, nz)

            base_cost = move_cost((cx, cy, cz), nxt)

            penalty = 0
            if nxt in penalty_cells:
                penalty += penalty_weight

            tg = g_score[current] + base_cost + penalty
            neighbor_state = (nx, ny, nz, nt)

            if tg < g_score.get(neighbor_state, float("inf")):
                g_score[neighbor_state] = tg
                came_from[neighbor_state] = current

                f = tg + heuristic(nxt, goal)

                heapq.heappush(open_heap, (f, neighbor_state))

    return [], float("inf"), -1


def generate_candidate_paths(
    env,
    start,
    goal,
    dynamic_obstacles,
    num_candidates=12,
    t_max=100
):
    candidates = []
    seen = set()

    penalty_sets = [set()]

    for k in range(num_candidates):
        penalty_cells = penalty_sets[k] if k < len(penalty_sets) else set()

        path, cost, arrival_t = astar_spacetime_with_penalty(
            env=env,
            start=start,
            goal=goal,
            dynamic_obstacles=dynamic_obstacles,
            penalty_cells=penalty_cells,
            penalty_weight=3.0 + k,
            t_max=t_max
        )

        if not path:
            continue

        path_tuple = tuple(path)

        if path_tuple not in seen:
            seen.add(path_tuple)

            candidates.append({
                "path": path,
                "cost": path_cost(path),
                "energy": path_energy(path),
                "arrival": len(path) - 1
            })

            middle_cells = path[1:-1]

            for step in range(2, max(3, len(middle_cells) // 4)):
                penalty_sets.append(set(middle_cells[::step]))

            penalty_sets.append(set(middle_cells))

    return candidates


# CSP Path Selection

In [6]:
def csp_select_multi_uav_paths(
    env,
    starts,
    goals,
    dynamic_obstacles,
    num_candidates=15,
    t_max_astar=100,

    # CSP constraints
    battery_limits=None,
    deadlines=None,

    # Objective weights
    alpha_travel=100,
    beta_makespan=10,
    gamma_energy=50,

    time_limit_seconds=60
):
    print("Generating candidate paths...")

    all_candidates = []
    n_uavs = len(starts)

    if battery_limits is None:
        battery_limits = [10_000] * n_uavs

    if deadlines is None:
        deadlines = [t_max_astar] * n_uavs

    for i in range(n_uavs):
        candidates = generate_candidate_paths(
            env=env,
            start=starts[i],
            goal=goals[i],
            dynamic_obstacles=dynamic_obstacles,
            num_candidates=num_candidates,
            t_max=t_max_astar
        )

        print(f"UAV {i + 1}: {len(candidates)} candidate paths")

        if not candidates:
            print(f"No candidate paths found for UAV {i + 1}")
            return [], {}

        all_candidates.append(candidates)

    horizon = max(
        max(c["arrival"] for c in candidates)
        for candidates in all_candidates
    )

    print(f"CSP horizon: {horizon}")

    model = cp_model.CpModel()

    # Decision variable:
    # choice[i] = selected candidate path index for UAV i
    choice = []

    for i in range(n_uavs):
        choice_var = model.NewIntVar(
            0,
            len(all_candidates[i]) - 1,
            f"choice_uav_{i}"
        )

        choice.append(choice_var)

    # Constraint 1:
    # Forbid UAV-UAV path conflicts
    for i in range(n_uavs):
        for j in range(i + 1, n_uavs):
            for pi, cand_i in enumerate(all_candidates[i]):
                for pj, cand_j in enumerate(all_candidates[j]):

                    if paths_conflict(
                        cand_i["path"],
                        cand_j["path"],
                        horizon
                    ):
                        model.AddForbiddenAssignments(
                            [choice[i], choice[j]],
                            [(pi, pj)]
                        )

    selected_cost = []
    selected_energy = []
    selected_arrival = []

    for i in range(n_uavs):
        costs = [
            int(round(c["cost"] * 100))
            for c in all_candidates[i]
        ]

        energies = [
            int(round(c["energy"] * 100))
            for c in all_candidates[i]
        ]

        arrivals = [
            c["arrival"]
            for c in all_candidates[i]
        ]

        cost_var = model.NewIntVar(
            0,
            max(costs),
            f"cost_uav_{i}"
        )

        energy_var = model.NewIntVar(
            0,
            max(energies),
            f"energy_uav_{i}"
        )

        arrival_var = model.NewIntVar(
            0,
            max(arrivals),
            f"arrival_uav_{i}"
        )

        model.AddElement(choice[i], costs, cost_var)
        model.AddElement(choice[i], energies, energy_var)
        model.AddElement(choice[i], arrivals, arrival_var)

        # Constraint 2:
        # Battery / energy limit
        battery_scaled = int(round(battery_limits[i] * 100))
        model.Add(energy_var <= battery_scaled)

        # Constraint 3:
        # Arrival deadline
        model.Add(arrival_var <= deadlines[i])

        selected_cost.append(cost_var)
        selected_energy.append(energy_var)
        selected_arrival.append(arrival_var)

    total_cost = model.NewIntVar(
        0,
        10_000_000,
        "total_cost"
    )

    total_energy = model.NewIntVar(
        0,
        10_000_000,
        "total_energy"
    )

    model.Add(total_cost == sum(selected_cost))
    model.Add(total_energy == sum(selected_energy))

    makespan = model.NewIntVar(
        0,
        horizon,
        "makespan"
    )

    model.AddMaxEquality(makespan, selected_arrival)

    # Objective:
    # Minimize travel cost + energy + makespan
    model.Minimize(
        alpha_travel * total_cost +
        gamma_energy * total_energy +
        beta_makespan * makespan
    )

    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = time_limit_seconds
    solver.parameters.num_search_workers = 8

    t0 = time.time()
    status = solver.Solve(model)
    runtime = time.time() - t0

    if status not in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
        print("No CSP solution found.")
        print("Solver status:", solver.StatusName(status))
        return [], {}

    selected_paths = []

    print("\n--- CSP Path Selection Results ---")
    print("Status:", solver.StatusName(status))
    print(f"Runtime: {runtime:.3f} seconds")
    print(f"Total travel cost: {solver.Value(total_cost) / 100:.2f}")
    print(f"Total energy: {solver.Value(total_energy) / 100:.2f}")
    print(f"Makespan: {solver.Value(makespan)}")

    for i in range(n_uavs):
        selected_idx = solver.Value(choice[i])
        selected = all_candidates[i][selected_idx]

        selected_paths.append(selected["path"])

        print(f"\nUAV {i + 1}")
        print(f"  Selected candidate: {selected_idx}")
        print(f"  Path length: {len(selected['path'])}")
        print(f"  Arrival time: {selected['arrival']}")
        print(f"  Travel cost: {selected['cost']:.2f}")
        print(f"  Energy: {selected['energy']:.2f}")
        print(f"  Battery limit: {battery_limits[i]}")
        print(f"  Deadline: {deadlines[i]}")

    metrics = {
        "runtime": runtime,
        "total_cost": solver.Value(total_cost) / 100,
        "total_energy": solver.Value(total_energy) / 100,
        "makespan": solver.Value(makespan),
        "path_costs": [path_cost(p) for p in selected_paths],
        "path_energies": [path_energy(p) for p in selected_paths],
        "path_lengths": [len(p) for p in selected_paths],
        "arrivals": [len(p) - 1 for p in selected_paths],
        "near_miss_count": near_miss_count(selected_paths, horizon),
        "horizon": horizon
    }

    print(f"\nNear-miss count: {metrics['near_miss_count']}")

    return selected_paths, metrics


# Visualozation

In [7]:
def plot_3d_paths(
    env,
    paths,
    dynamic_obstacles,
    starts,
    goals,
    title="Multi-UAV Path Planning"
):
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection="3d")

    x, y, z = np.where(env.grid == 1)
    ax.scatter(
        x,
        y,
        z,
        c="red",
        marker="s",
        alpha=0.1,
        s=40,
        label="Static NFZ"
    )

    for obs in dynamic_obstacles:
        ox, oy, oz = zip(*obs.waypoints)
        ax.plot(
            ox,
            oy,
            oz,
            "--",
            alpha=0.5,
            label=f"Trajectory {obs.name}"
        )

    colors = ["blue", "green", "orange", "purple", "black"]

    for i, path in enumerate(paths):
        if path:
            px, py, pz = zip(*path)
            ax.plot(
                px,
                py,
                pz,
                "-o",
                markersize=3,
                linewidth=2,
                color=colors[i % len(colors)],
                label=f"UAV {i + 1} Path"
            )

    for i, start in enumerate(starts):
        ax.scatter(
            *start,
            c=colors[i % len(colors)],
            s=150,
            marker="^",
            label=f"UAV {i + 1} Start"
        )

    for i, goal in enumerate(goals):
        ax.scatter(
            *goal,
            c=colors[i % len(colors)],
            s=150,
            marker="*",
            label=f"UAV {i + 1} Goal"
        )

    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.legend()

    plt.show()


def generate_uav_animation_gif(
    env,
    uav_paths_list,
    env_dynamic_obstacles,
    starts_list,
    goals_list,
    uav_colors,
    title,
    filename
):
    if not uav_paths_list:
        print("No UAV paths to animate.")
        return

    t_max = max(len(p) for p in uav_paths_list) - 1

    all_extended_uav_paths = []

    for path_uav, goal_uav in zip(uav_paths_list, goals_list):
        extended_path = list(path_uav)

        while len(extended_path) <= t_max:
            extended_path.append(goal_uav)

        all_extended_uav_paths.append(extended_path)

    max_animation_frames = t_max + 1

    fig_anim = plt.figure(figsize=(12, 8))
    ax_anim = fig_anim.add_subplot(111, projection="3d")

    x_static, y_static, z_static = np.where(env.grid == 1)

    ax_anim.scatter(
        x_static,
        y_static,
        z_static,
        c="red",
        marker="s",
        alpha=0.1,
        s=40,
        label="Static NFZ"
    )

    for obs in env_dynamic_obstacles:
        ox, oy, oz = zip(*obs.waypoints)

        ax_anim.plot(
            ox,
            oy,
            oz,
            ":",
            color="gray",
            alpha=0.7,
            label=f"Trajectory {obs.name}"
        )

    uav_dots = []
    uav_lines = []

    for i, path_uav in enumerate(all_extended_uav_paths):
        color = uav_colors[i % len(uav_colors)]

        dot, = ax_anim.plot(
            [],
            [],
            [],
            "o",
            markersize=8,
            color=color,
            label=f"UAV {i + 1}"
        )

        line, = ax_anim.plot(
            [],
            [],
            [],
            "-.",
            color=color,
            linewidth=1
        )

        uav_dots.append(dot)
        uav_lines.append(line)

        ax_anim.scatter(*starts_list[i], c=color, s=150, marker="^")
        ax_anim.scatter(*goals_list[i], c=color, s=150, marker="*")

    dyn_obs_dots = []

    for i, obs in enumerate(env_dynamic_obstacles):
        dot, = ax_anim.plot(
            [],
            [],
            [],
            "X",
            markersize=10,
            color="purple",
            label=f"Dynamic Obs {i + 1}"
        )

        dyn_obs_dots.append(dot)

    ax_anim.set_xlim([0, env.shape[0]])
    ax_anim.set_ylim([0, env.shape[1]])
    ax_anim.set_zlim([0, env.shape[2]])

    ax_anim.set_xlabel("X")
    ax_anim.set_ylabel("Y")
    ax_anim.set_zlabel("Z")
    ax_anim.set_title(title)
    ax_anim.legend()

    def update(frame):
        artists = []

        for i, path_uav_extended in enumerate(all_extended_uav_paths):
            current_uav_pos = path_uav_extended[frame]

            uav_dots[i].set_data(
                [current_uav_pos[0]],
                [current_uav_pos[1]]
            )

            uav_dots[i].set_3d_properties(
                [current_uav_pos[2]]
            )

            artists.append(uav_dots[i])

            uav_lines[i].set_data(
                [p[0] for p in path_uav_extended[:frame + 1]],
                [p[1] for p in path_uav_extended[:frame + 1]]
            )

            uav_lines[i].set_3d_properties(
                [p[2] for p in path_uav_extended[:frame + 1]]
            )

            artists.append(uav_lines[i])

        for i, obs in enumerate(env_dynamic_obstacles):
            obs_pos = obs.position_at(frame)

            dyn_obs_dots[i].set_data(
                [obs_pos[0]],
                [obs_pos[1]]
            )

            dyn_obs_dots[i].set_3d_properties(
                [obs_pos[2]]
            )

            artists.append(dyn_obs_dots[i])

        fig_anim.suptitle(
            f"{title} - Time: {frame}",
            fontsize=16
        )

        return artists

    anim = animation.FuncAnimation(
        fig_anim,
        update,
        frames=range(max_animation_frames),
        interval=200,
        blit=True
    )

    gif_filename = filename

    print(f"Saving animation as '{gif_filename}'...")
    anim.save(gif_filename, writer="pillow", fps=10)
    print(f"Animation '{gif_filename}' saved!")

    plt.close(fig_anim)

    mp4_filename = filename.replace(".gif", ".mp4")

    print(f"Converting '{gif_filename}' to '{mp4_filename}'...")

    try:
        clip = VideoFileClip(gif_filename)
        clip.write_videofile(
            mp4_filename,
            codec="libx264",
            audio_codec="aac"
        )

        print(f"Successfully converted to '{mp4_filename}'!")

    except Exception as e:
        print(f"Error converting GIF to MP4: {e}")


# Tesing Scenario

In [8]:
env_multi_uav = GridEnvironment(shape=(20, 20, 10))

# Static no-fly zones
env_multi_uav.add_obstacle_block((5, 10), (5, 15), (0, 5))
env_multi_uav.add_obstacle_block((12, 17), (2, 8), (2, 7))


# Dynamic obstacle 1
obs1_waypoints_multi = [
    (2,5,2), (3,5,2), (4,5,2), (4,6,2), (4,7,2), (3,7,2), (2,7,2), (2,6,2),
    (2,5,3), (3,5,3), (4,5,3), (4,6,3), (4,7,3), (3,7,3), (2,7,3), (2,6,3),
    (2,5,4), (3,5,4), (4,5,4), (4,6,4), (4,7,4), (3,7,4), (2,7,4), (2,6,4),
    (1,5,4), (0,5,4), (0,6,4), (0,7,4), (1,7,4), (1,6,4)
]

# Dynamic obstacle 2
obs2_waypoints_multi = [
    (10,9,2), (10,9,3), (10,9,4), (10,9,5), (10,9,4), (10,9,3),
    (10,10,3), (10,10,4), (10,10,3),
    (10,11,3), (10,11,4), (10,11,3),
    (9,11,3), (9,11,4), (9,11,3),
    (8,11,3), (8,11,4), (8,11,3)
]

env_hazards_multi = [
    DynamicObstacle(obs1_waypoints_multi, loop=True, name="Dyn_West"),
    DynamicObstacle(obs2_waypoints_multi, loop=True, name="Dyn_Center")
]

starts_multi = [
    (0, 0, 1),
    (0, 2, 1),
    (2, 0, 1)
]

goals_multi = [
    (18, 18, 4),
    (15, 3, 6),
    (5, 18, 8)
]



# CSP Scenario

In [ ]:
battery_limits = [
    90,
    80,
    85
]

deadlines = [
    35,
    35,
    40
]

uav_results_multi, metrics = csp_select_multi_uav_paths(
    env=env_multi_uav,
    starts=starts_multi,
    goals=goals_multi,
    dynamic_obstacles=env_hazards_multi,

    num_candidates=20,
    t_max_astar=100,

    battery_limits=battery_limits,
    deadlines=deadlines,

    alpha_travel=100,
    beta_makespan=10,
    gamma_energy=50,

    time_limit_seconds=60
)

Generating candidate paths...
UAV 1: 4 candidate paths


# Plotting and animation

In [ ]:
if uav_results_multi:
    plot_3d_paths(
        env=env_multi_uav,
        paths=uav_results_multi,
        dynamic_obstacles=env_hazards_multi,
        starts=starts_multi,
        goals=goals_multi,
        title="Hybrid A* + CP-SAT Multi-UAV Path Planning"
    )

    generate_uav_animation_gif(
        env=env_multi_uav,
        uav_paths_list=uav_results_multi,
        env_dynamic_obstacles=env_hazards_multi,
        starts_list=starts_multi,
        goals_list=goals_multi,
        uav_colors=["royalblue", "limegreen", "darkorange"],
        title="Hybrid A* + CP-SAT Multi-UAV Path Planning",
        filename="hybrid_csp_multi_uav_simulation.gif"
    )

else:
    print("No paths found.")


# Metrics

In [ ]:
print("\n--- Final Metrics ---")

if metrics:
    print(f"Runtime: {metrics['runtime']:.3f} seconds")
    print(f"Total travel cost: {metrics['total_cost']:.2f}")
    print(f"Total energy: {metrics['total_energy']:.2f}")
    print(f"Makespan: {metrics['makespan']}")
    print(f"Near-miss count: {metrics['near_miss_count']}")

    for i in range(len(uav_results_multi)):
        print(f"\nUAV {i + 1}")
        print(f"  Path length: {metrics['path_lengths'][i]}")
        print(f"  Arrival time: {metrics['arrivals'][i]}")
        print(f"  Path cost: {metrics['path_costs'][i]:.2f}")
        print(f"  Energy: {metrics['path_energies'][i]:.2f}")
        print(f"  Battery limit: {battery_limits[i]}")
        print(f"  Deadline: {deadlines[i]}")